<h1><strong>Problem Statement</strong></h1>


<h2>Business Context</h2>


<p>Renewable energy sources play an increasingly important role in the global energy mix, as the effort to reduce the environmental impact of energy production increases.</p>
<p>Out of all the renewable energy alternatives, wind energy is one of the most developed technologies worldwide. The U.S Department of Energy has put together a guide to achieving operational efficiency using predictive maintenance practices.</p>
<p>Predictive maintenance uses sensor information and analysis methods to measure and predict degradation and future component capability. The idea behind predictive maintenance is that failure patterns are predictable and if component failure can be predicted accurately and the component is replaced before it fails, the costs of operation and maintenance will be much lower.</p>
<p>The sensors fitted across different machines involved in the process of energy generation collect data related to various environmental factors (temperature, humidity, wind speed, etc.) and additional features related to various parts of the wind turbine (gearbox, tower, blades, break, etc.).</p>


<h2>Objective</h2>


<p>“ReneWind” is a company working on improving the machinery/processes involved in the production of wind energy using machine learning and has collected data of generator failure of wind turbines using sensors. They have shared a ciphered version of the data, as the data collected through sensors is confidential (the type of data collected varies with companies). Data has 40 predictors, 20000 observations in the training set and 5000 in the test set.</p>
<p>The objective is to build various classification models, tune them, and find the best one that will help identify failures so that the generators could be repaired before failing/breaking to reduce the overall maintenance cost.
The nature of predictions made by the classification model will translate as follows:</p>
<ul>
<li>True positives (TP) are failures correctly predicted by the model. These will result in repairing costs.</li>
<li>False negatives (FN) are real failures where there is no detection by the model. These will result in replacement costs.</li>
<li>False positives (FP) are detections where there is no failure. These will result in inspection costs.</li>
</ul>
<p>It is given that the cost of repairing a generator is much less than the cost of replacing it, and the cost of inspection is less than the cost of repair.</p>
<p>“1” in the target variables should be considered as “failure” and “0” represents “No failure”.</p>


<h2>Data Description</h2>


<p>The data provided is a transformed version of the original data which was collected using sensors.</p>
<ul>
<li>Train.csv - To be used for training and tuning of models.</li>
<li>Test.csv - To be used only for testing the performance of the final best model.</li>
</ul>
<p>Both the datasets consist of 40 predictor variables and 1 target variable.</p>


<h1><strong>Installing and Importing the necessary libraries</strong></h1>


In [ ]:
# Installing the libraries with the specified version
!pip install --no-deps tensorflow==2.19.0 scikit-learn==1.6.1 matplotlib===3.10.0 seaborn==0.13.2 numpy==2.0.2 pandas==2.2.2 -q --user --no-warn-script-location


In [ ]:
# Library for data manipulation and analysis.
import pandas as pd
# Fundamental package for scientific computing.
import numpy as np
#splitting datasets into training and testing sets.
from sklearn.model_selection import train_test_split
#Imports tools for data preprocessing including label encoding, one-hot encoding, and standard scaling
from sklearn.preprocessing import LabelEncoder, OneHotEncoder,StandardScaler
#Imports a class for imputing missing values in datasets.
from sklearn.impute import SimpleImputer
#Imports the Matplotlib library for creating visualizations.
import matplotlib.pyplot as plt
# Imports the Seaborn library for statistical data visualization.
import seaborn as sns
# Time related functions.
import time
#Imports functions for evaluating the performance of machine learning models
from sklearn.metrics import confusion_matrix, f1_score,accuracy_score, recall_score, precision_score, classification_report

#Imports the tensorflow,keras and layers.
import tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dense, Input, Dropout,BatchNormalization
from tensorflow.keras import backend

# to suppress unnecessary warnings
import warnings
warnings.filterwarnings("ignore")


<p><strong>Note</strong>:</p>
<ul>
<li>After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.</li>
<li>On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in <em><strong>this notebook</strong></em>.</li>
</ul>


<h1><strong>Loading the Data</strong></h1>


In [ ]:
# uncomment and run the following lines for Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
data=pd.read_csv('Train.csv')
data_test=pd.read_csv('Test.csv')


In [ ]:
df = data.copy()
df_test = data_test.copy()


<h1><strong>Data Overview</strong></h1>


In [ ]:
df.head()


In [ ]:
df_test.head()


In [ ]:
df.shape


In [ ]:
df_test.shape


In [ ]:
df.info()


<p>All of them are indicators are float</p>


In [ ]:
df_test.info()


In [ ]:
df.duplicated().sum()


In [ ]:
df_test.duplicated().sum()


In [ ]:
df.isnull().sum()


<p>There are 18 null values in V1 and V2.</p>


In [ ]:
df_test.isnull().sum()


<p>there are 5 null values in V1 and 6 in V2 in test set.</p>


In [ ]:
df['Target'].value_counts()


In [ ]:
df_test['Target'].value_counts()


In [ ]:
df.describe()


In [ ]:
df_test.describe()


<h1><strong>Exploratory Data Analysis</strong></h1>


<h2>Univariate analysis</h2>


In [ ]:
# Function to create labeled barplots

def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot


In [ ]:
# Function to plot a boxplot and a histogram along the same scale.

def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram


In [ ]:
### Function to plot distributions

def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()


In [ ]:
# function to plot stacked bar chart

def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 1, 5))
    plt.legend(
        loc="lower left",
        frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()


In [ ]:
for col in data.columns[:-1]: # Exclude the 'Target' column
    histogram_boxplot(data, col)


<h3>Summary of Univariate Analysis Observations</h3><p>The univariate analysis of features V1 through V40 revealed several key insights into their distributions, skewness, and the presence of outliers:</p>
<ul>
<li><p><strong>General Distribution</strong>: Most features appear to be continuous and exhibit a somewhat normal or slightly skewed distribution. The central tendency (mean and median) for many features seems to be close to zero, suggesting that the data might have been standardized or transformed.</p>
</li>
<li><p><strong>Skewness</strong>: Many features show varying degrees of skewness. Some columns, like V3, V11, V15, V26, V35, and V36, exhibit a noticeable positive skew, indicating a longer tail towards higher values. Conversely, features such as V4, V19, V32, and V34 display a negative skew, with a longer tail towards lower values.</p>
</li>
<li><p><strong>Outliers</strong>: The boxplots clearly show the presence of outliers in almost all feature columns. These outliers are visible as individual points extending beyond the whiskers of the boxplots. The extent and magnitude of these outliers vary across features, with some columns showing more extreme outliers than others. For example, V4, V14, V19, V32, and V34 appear to have a significant number of outliers on both ends of their distributions.</p>
</li>
<li><p><strong>Symmetry</strong>: Some features, like V5, V6, V7, V8, V9, V10, V12, V13, V16, V17, V18, V20, V21, V22, V23, V24, V25, V27, V28, V29, V30, V31, V33, V37, V38, V39, and V40, show relatively symmetric distributions, with their mean and median being quite close. However, even these features often contain outliers.</p>
</li>
<li><p><strong>Range</strong>: The ranges of values for each feature vary significantly, as expected with diverse sensor data. Some features span a wider range of values, while others are more concentrated.</p>
</li>
</ul>
<p>These observations highlight the need for careful consideration during feature engineering and model building. The presence of skewness might necessitate transformations (e.g., log transformation) to achieve a more normal distribution, which can benefit certain machine learning models. The numerous outliers also suggest that robust scaling methods or outlier treatment strategies (e.g., capping, removal) might be beneficial for improving model performance and stability.</p>


<h2>Bivariate Analysis</h2>


In [ ]:
for col in data.columns[:-1]:
    distribution_plot_wrt_target(data, col, 'Target')


In [ ]:
cols_list = data.columns.tolist()

plt.figure(figsize=(20, 20))
sns.heatmap(
    data[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral"
)
plt.title('Correlation Heatmap of Features and Target')
plt.show()


<p>There are some highly positively correlated features with each other (e.g., V15 and V7 have a correlation of 0.87) and some negatively correlated features with each other (e.g., V14 and V2 have a correlation of -0.85).
The correlation of any particular feature with the Target variable is relatively low, indicating no strong linear relationships between individual features and the target.</p>


<h3>Summary of Bivariate Analysis Observations</h3><ul>
<li><p><strong>Discriminating Features</strong>: Some features show a clear difference in their distribution between the '0' (no failure) and '1' (failure) target classes. For example, features like <strong>V3, V11, V15, V26, V35, and V36</strong> tend to have noticeably different mean/median values or spread for the two target classes, indicating they could be strong predictors. For instance, the boxplots for these features often show that the median for 'Target=1' is significantly higher or lower than for 'Target=0', and their histograms might show less overlap between the two target distributions.</p>
</li>
<li><p><strong>Less Discriminating Features</strong>: Conversely, many features appear to have very similar distributions for both 'Target=0' and 'Target=1'. For these features, the histograms for both classes largely overlap, and the boxplots show similar median and quartile ranges. This suggests that these features might have limited individual predictive power for distinguishing between failure and non-failure instances.</p>
</li>
<li><p><strong>Impact of Outliers</strong>: The presence of outliers, as observed in the univariate analysis, also seems to affect the separation between the target classes for some features. In some cases, outliers might contribute to the distinctiveness of a feature's distribution for one target class over the other, while in other cases, they might obscure underlying patterns.</p>
</li>
<li><p><strong>Imbalanced Data Consideration</strong>: Given that the 'Target' variable is imbalanced (18890 'no failure' vs. 1110 'failure'), the visual differences in distributions, particularly for the 'failure' class, might appear subtle due to its smaller representation. However, even subtle shifts in distribution for the minority class can be indicative of predictive strength.</p>
</li>
</ul>


<h1><strong>Data Preprocessing</strong></h1>


In [ ]:
X = df.drop(columns = ["Target"] , axis=1)

Y = df["Target"]


In [ ]:
# Splitting train dataset into training and validation sets
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=1, stratify=Y)


In [ ]:
# Check the shape of X_train and y_train
print(X_train.shape)
print(Y_train.shape)


In [ ]:
# Check the shape of X_val and y_val
print(X_val.shape)
print(Y_val.shape)


In [ ]:
# Divide test data into X_test and y_test

# Drop the 'Target' column for X
X_test = data_test.drop(columns = ['Target'] , axis= 1)

# Retain only 'Target' column for y
Y_test = data_test["Target"]


In [ ]:
#Printing the shapes.
print(X_test.shape,Y_test.shape)


In [ ]:
imputer_mode = SimpleImputer(strategy="median")
X_train[['V1', 'V2']] = imputer_mode.fit_transform(X_train[['V1', 'V2']])
X_val[['V1', 'V2']] = imputer_mode.fit_transform(X_val[['V1', 'V2']])
X_test[['V1', 'V2']] = imputer_mode.fit_transform(df_test[['V1', 'V2']])


In [ ]:
X_train.isnull().sum()


In [ ]:
X_val.isnull().sum()


In [ ]:
X_test.isnull().sum()


In [ ]:
# Initialize StandardScaler
scaler = StandardScaler()
# Fit the scaler on the training data and transform it
# This calculates the mean and standard deviation from the training data and applies the transformation
X_train = scaler.fit_transform(X_train)

# Transform the validation data using the scaler fitted on the training data
# This ensures that the validation data is scaled using the same parameters as the training data, preventing data leakage
X_val = scaler.transform(X_val)

# Transform the test data using the scaler fitted on the training data
# This ensures consistency in scaling across all datasets
X_test = scaler.transform(X_test)


In [ ]:
# Convert target variables to NumPy arrays for compatibility with neural network libraries
Y_train = Y_train.to_numpy()
Y_val = Y_val.to_numpy()
Y_test = Y_test.to_numpy()
print("y_train shape : ", Y_train.shape)
print("y_val shape : ", Y_val.shape)
print("y_test shape : ", Y_test.shape)


<h1><strong>Model Building</strong></h1>


<h2>Model Evaluation Criterion</h2>


<p>Write down the model evaluation criterion with rationale</p>


<h2>Utility Functions</h2>


In [ ]:
def plot(history, name):
    """
    Function to plot loss/accuracy

    history: an object which stores the metrics and losses.
    name: can be one of Loss or Accuracy
    """
    fig, ax = plt.subplots() #Creating a subplot with figure and axes.
    plt.plot(history.history[name]) #Plotting the train accuracy or train loss
    plt.plot(history.history['val_'+name]) #Plotting the validation accuracy or validation loss

    plt.title('Model ' + name.capitalize()) #Defining the title of the plot.
    plt.ylabel(name.capitalize()) #Capitalizing the first letter.
    plt.xlabel('Epoch') #Defining the label for the x-axis.
    fig.legend(['Train', 'Validation'], loc="outside right upper") #Defining the legend, loc controls the position of the legend.
# defining a function to compute different metrics to check performance of a classification model built using statsmodels

def model_performance_classification(
    model, predictors, target, threshold=0.5
):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    threshold: threshold for classifying the observation as class 1
    """

    # checking which probabilities are greater than threshold
    pred = model.predict(predictors) > threshold
    # pred_temp = model.predict(predictors) > threshold
    # # rounding off the above values to get classes
    # pred = np.round(pred_temp)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred, average='weighted')  # to compute Recall
    precision = precision_score(target, pred, average='weighted')  # to compute Precision
    f1 = f1_score(target, pred, average='weighted')  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1 Score": f1,},
        index=[0],
    )

    return df_perf


In [ ]:
# Since the class is imabalnced in the data, we'll use class weights in all the models
# Calculate class weights for imbalanced dataset. This gives higher weight to the minority class.
cw = (Y_train.shape[0]) / np.bincount(Y_train.astype(int))

# Create a dictionary mapping class indices to their respective class weights
cw_dict = {}
for i in range(cw.shape[0]):
    cw_dict[i] = cw[i]

cw_dict


<h2>Initial Model Building (Model 0)</h2>


<ul>
<li>Let's start with a neural network consisting of<ul>
<li>just one hidden layer</li>
<li>activation function of ReLU</li>
<li>SGD as the optimizer</li>
</ul>
</li>
</ul>


In [ ]:
# defining the batch size and # epochs upfront as we'll be using the same values for all models
epochs = 25
batch_size = 32


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
# Initializing the sequential model
model_0 = Sequential()

# Add the first hidden layer with 7 neurons, ReLU activation, and input dimension matching the number of features in X_train
model_0.add(Dense(7, activation="relu",input_dim=X_train.shape[1]))

# Add the output layer with 1 neuron and sigmoid activation for binary classification probability output
model_0.add(Dense(1, activation="sigmoid"))


In [ ]:
model_0.summary()


In [ ]:
optimizer = tf.keras.optimizers.SGD()    # defining SGD as the optimizer to be used
model_0.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_0.fit(X_train, Y_train, validation_data=(X_val,Y_val), batch_size=batch_size, epochs=epochs, class_weight=cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_0_train_perf = model_performance_classification(model_0, X_train, Y_train)
model_0_train_perf


In [ ]:
model_0_valid_perf = model_performance_classification(model_0, X_val, Y_val)
model_0_valid_perf


<h1><strong>Model Performance Improvement</strong></h1>


<h2>Model 1</h2>


<p>Adding momentum</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_1 = Sequential()
model_1.add(Dense(7,activation="relu",input_dim=X_train.shape[1]))
model_1.add(Dense(1,activation="sigmoid"))


In [ ]:
model_1.summary()


In [ ]:
optimizer = tf.keras.optimizers.SGD(momentum=0.9)    # defining SGD as the optimizer to be used
model_1.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_1.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=batch_size, epochs=epochs,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_1_train_perf = model_performance_classification(model_1, X_train, Y_train)
model_1_train_perf


In [ ]:
model_1_valid_perf = model_performance_classification(model_1, X_val, Y_val)
model_1_valid_perf


<h2>Model 2</h2>


<p>Adding momentum and one more hidden layers</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_2 = Sequential()
model_2.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_2.add(Dense(7,activation="relu"))
model_2.add(Dense(1,activation="sigmoid"))


In [ ]:
model_2.summary()


In [ ]:
optimizer = tf.keras.optimizers.SGD(momentum=0.9)    # defining SGD as the optimizer to be used
model_2.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_2.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=batch_size, epochs=epochs,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_2_train_perf = model_performance_classification(model_2, X_train, Y_train)
model_2_train_perf


In [ ]:
model_2_valid_perf = model_performance_classification(model_2, X_val, Y_val)
model_2_valid_perf


<h2>Model 3</h2>


<p>Increasing Epochs</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_3 = Sequential()
model_3.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_3.add(Dense(7,activation="relu"))
model_3.add(Dense(1,activation="sigmoid"))


In [ ]:
model_3.summary()


In [ ]:
optimizer = tf.keras.optimizers.SGD(momentum=0.9)    # defining SGD as the optimizer to be used
model_3.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_3.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=batch_size, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_3_train_perf = model_performance_classification(model_3, X_train, Y_train)
model_3_train_perf


In [ ]:
model_3_valid_perf = model_performance_classification(model_3, X_val, Y_val)
model_3_valid_perf


<h2>Model 4</h2>


<p>Increasing batch size</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_4 = Sequential()
model_4.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_4.add(Dense(7,activation="relu"))
model_4.add(Dense(1,activation="sigmoid"))


In [ ]:
model_3.summary()


In [ ]:
optimizer = tf.keras.optimizers.SGD(momentum=0.9)    # defining SGD as the optimizer to be used
model_4.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_4.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_4_train_perf = model_performance_classification(model_4, X_train, Y_train)
model_4_train_perf


In [ ]:
model_4_valid_perf = model_performance_classification(model_4, X_val, Y_val)
model_4_valid_perf


<h2>Model 5</h2>


<p>Going to use adam optimizer</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_5 = Sequential()
model_5.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_5.add(Dense(7,activation="relu"))
model_5.add(Dense(1,activation="sigmoid"))


In [ ]:
model_5.summary()


In [ ]:
optimizer = tf.keras.optimizers.Adam()    # defining Adam as the optimizer to be used
model_5.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_5.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_5_train_perf = model_performance_classification(model_5, X_train, Y_train)
model_5_train_perf


In [ ]:
model_5_valid_perf = model_performance_classification(model_5, X_val, Y_val)
model_5_valid_perf


<h2>Model 6</h2>


<p>Going to use adam optimizer with dropout</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_6 = Sequential()
model_6.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_6.add(Dropout(0.4))
model_6.add(Dense(7,activation="relu"))
model_6.add(Dropout(0.2))
model_6.add(Dense(1,activation="sigmoid"))


In [ ]:
model_6.summary()


In [ ]:
optimizer = tf.keras.optimizers.Adam()    # defining Adam as the optimizer to be used
model_6.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_6.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_6_train_perf = model_performance_classification(model_6, X_train, Y_train)
model_6_train_perf


In [ ]:
model_6_valid_perf = model_performance_classification(model_6, X_val, Y_val)
model_6_valid_perf


<h2>Model 7</h2>


<p>Going to use adam optimizer and smaller batch size with drop out</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_7 = Sequential()
model_7.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_7.add(Dropout(0.4))
model_7.add(Dense(7,activation="relu"))
model_7.add(Dropout(0.2))
model_7.add(Dense(1,activation="sigmoid"))


In [ ]:
model_7.summary()


In [ ]:
optimizer = tf.keras.optimizers.Adam()    # defining Adam as the optimizer to be used
model_7.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_7.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=32, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_7_train_perf = model_performance_classification(model_7, X_train, Y_train)
model_7_train_perf


In [ ]:
model_7_valid_perf = model_performance_classification(model_7, X_val, Y_val)
model_7_valid_perf


<h2>Model 8</h2>


<p>Going to use adam optimizer and smaller batch size with no drop out as drop out is not improving the performance</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_8 = Sequential()
model_8.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_8.add(Dense(7,activation="relu"))
model_8.add(Dense(1,activation="sigmoid"))


In [ ]:
model_8.summary()


In [ ]:
optimizer = tf.keras.optimizers.Adam()    # defining Adam as the optimizer to be used
model_8.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_8.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=32, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_8_train_perf = model_performance_classification(model_8, X_train, Y_train)
model_8_train_perf


In [ ]:
model_8_valid_perf = model_performance_classification(model_8, X_val, Y_val)
model_8_valid_perf


<h2>Model 9</h2>


<p>Changing first activation layer from relu to leaky_relu</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_9 = Sequential()
model_9.add(Dense(14,activation="leaky_relu",input_dim=X_train.shape[1]))
model_9.add(Dense(7,activation="leaky_relu"))
model_9.add(Dense(1,activation="sigmoid"))


In [ ]:
model_9.summary()


In [ ]:
optimizer = tf.keras.optimizers.Adam()    # defining Adam as the optimizer to be used
model_9.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_9.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_9_train_perf = model_performance_classification(model_9, X_train, Y_train)
model_9_train_perf


In [ ]:
model_9_valid_perf = model_performance_classification(model_9, X_val, Y_val)
model_9_valid_perf


<h2>Model 10</h2>


<p>Going to apply batch normalization</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_10 = Sequential()
model_10.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_10.add(BatchNormalization())
model_10.add(Dense(7,activation="relu"))
model_10.add(BatchNormalization())
model_10.add(Dense(1,activation="sigmoid"))


In [ ]:
model_10.summary()


In [ ]:
optimizer = tf.keras.optimizers.Adam()    # defining Adam as the optimizer to be used
model_10.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_10.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_10_train_perf = model_performance_classification(model_10, X_train, Y_train)
model_10_train_perf


In [ ]:
model_10_valid_perf = model_performance_classification(model_10, X_val, Y_val)
model_10_valid_perf


<h2>Model 11</h2>


<p>Going to apply batch normalization but going to change optimizer back to SGD with momentum</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_11 = Sequential()
model_11.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_11.add(BatchNormalization())
model_11.add(Dense(7,activation="relu"))
model_11.add(BatchNormalization())
model_11.add(Dense(1,activation="sigmoid"))


In [ ]:
model_11.summary()


In [ ]:
optimizer = tf.keras.optimizers.SGD(momentum=0.9)    # defining SGD as the optimizer to be used
model_11.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_11.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_11_train_perf = model_performance_classification(model_11, X_train, Y_train)
model_11_train_perf


In [ ]:
model_11_valid_perf = model_performance_classification(model_11, X_val, Y_val)
model_11_valid_perf


<h2>Model 12</h2>


<p>Let's initialize the weights using He normalization</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_12 = Sequential()
model_12.add(Dense(14,activation="relu",kernel_initializer="he_normal",input_dim=X_train.shape[1]))
model_12.add(BatchNormalization())
model_12.add(Dense(7,activation="relu",kernel_initializer="he_normal"))
model_12.add(BatchNormalization())
model_12.add(Dense(1,activation="sigmoid",kernel_initializer="he_normal"))


In [ ]:
model_12.summary()


In [ ]:
optimizer = tf.keras.optimizers.Adam() # defining Adam as the optimizer to be used
model_12.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_12.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_12_train_perf = model_performance_classification(model_11, X_train, Y_train)
model_12_train_perf


In [ ]:
model_12_valid_perf = model_performance_classification(model_12, X_val, Y_val)
model_12_valid_perf


<h2>Model 13</h2>


<p>Adding a learning rate</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_13 = Sequential()

model_13.add(Dense(21, activation="leaky_relu",kernel_initializer='he_normal', input_dim=X_train.shape[1]))

model_13.add(Dense(14, activation="relu", kernel_initializer='he_normal'))

model_13.add(Dense(7, activation="relu"))

model_13.add(Dense(1, activation="sigmoid"))


In [ ]:
model_13.summary()


In [ ]:
# Define Adam as the optimizer to be used
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

# Recall is the chosen metric to measure
model_13.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_13.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_13_train_perf = model_performance_classification(model_13, X_train, Y_train)
model_13_train_perf


In [ ]:
model_13_valid_perf = model_performance_classification(model_13, X_val, Y_val)
model_13_valid_perf


<h2>Model 14</h2>


<p>Increasing Epochs</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_14 = Sequential()
model_14.add(Dense(14,activation="relu",input_dim=X_train.shape[1]))
model_14.add(Dense(7,activation="relu"))
model_14.add(Dense(1,activation="sigmoid"))


In [ ]:
model_14.summary()


In [ ]:
# Define Adam as the optimizer to be used
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

# Recall is the chosen metric to measure
model_14.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_14.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=batch_size, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_14_train_perf = model_performance_classification(model_14, X_train, Y_train)
model_14_train_perf


In [ ]:
model_14_valid_perf = model_performance_classification(model_14, X_val, Y_val)
model_14_valid_perf


<p><em>italicized text</em>## Model 14</p>


<h2>Model 15</h2>


<p>Increasing Epochs</p>


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
model_15 = Sequential()
model_15.add(Dense(21,activation="relu",kernel_initializer="he_normal",input_dim=X_train.shape[1]))
model_15.add(BatchNormalization())
model_15.add(Dense(7,activation="relu",kernel_initializer="he_normal"))
model_15.add(BatchNormalization())
model_15.add(Dense(7,activation="relu",kernel_initializer="he_normal"))
model_15.add(BatchNormalization())
model_15.add(Dense(1,activation="sigmoid",kernel_initializer="he_normal"))


In [ ]:
model_15.summary()


In [ ]:
# Define Adam as the optimizer to be used
optimizer = tf.keras.optimizers.SGD(momentum=0.9)

# Recall is the chosen metric to measure
model_15.compile(loss='binary_crossentropy', optimizer=optimizer, metrics = ['Recall'])


In [ ]:
start = time.time()
history = model_15.fit(X_train, Y_train, validation_data=(X_val,Y_val) , batch_size=64, epochs=50,class_weight = cw_dict)
end=time.time()


In [ ]:
plot(history,'loss')


In [ ]:
plot(history,'Recall')


In [ ]:
model_15_train_perf = model_performance_classification(model_15, X_train, Y_train)
model_15_train_perf


In [ ]:
model_15_valid_perf = model_performance_classification(model_15, X_val, Y_val)
model_15_valid_perf


<h1><strong>Model Performance Comparison and Final Model Selection</strong></h1>


<p>Now, in order to select the final model, we will compare the performances of all the models for the training and validation sets.</p>


In [ ]:
models_train_comp_df = pd.concat(
    [
        model_0_train_perf.T,
        model_1_train_perf.T,
        model_2_train_perf.T,
        model_3_train_perf.T,
        model_4_train_perf.T,
        model_5_train_perf.T,
        model_6_train_perf.T,
        model_7_train_perf.T,
        model_8_train_perf.T,
        model_9_train_perf.T,
        model_10_train_perf.T,
        model_11_train_perf.T,
        model_12_train_perf.T,
        model_13_train_perf.T,
        model_14_train_perf.T,
        model_15_train_perf.T
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Model 0 (SGD, 1-Layer(7-ReLU))",
    "Model 1 (SGD-Momentum, 1-Layer(7-ReLU))",
    "Model 2 (SGD-Momentum, 2-Layers(14-ReLU, 7-ReLU))",
    "Model 3 (SGD-Momentum, 2-Layers(14-ReLU, 7-ReLU), Epochs=50)",
    "Model 4 (SGD-Momentum, 2-Layers(14-ReLU, 7-ReLU), Epochs=50, Batch=64)",
    "Model 5 (Adam, 2-Layers(14-ReLU, 7-ReLU), Epochs=50, Batch=64)",
    "Model 6 (Adam, 2-Layers(14-ReLU, 7-ReLU), Dropout(0.4, 0.2), Epochs=50, Batch=64)",
    "Model 7 (Adam, 2-Layers(14-ReLU, 7-ReLU), Dropout(0.4, 0.2), Epochs=50, Batch=32)",
    "Model 8 (Adam, 2-Layers(14-ReLU, 7-ReLU), Epochs=50, Batch=32)",
    "Model 9 (Adam, 2-Layers(14-LeakyReLU, 7-LeakyReLU), Epochs=50, Batch=64)",
    "Model 10 (Adam, 2-Layers(14-ReLU, 7-ReLU), BatchNorm, Epochs=50, Batch=64)",
    "Model 11 (SGD-Momentum, 2-Layers(14-ReLU, 7-ReLU), BatchNorm, Epochs=50, Batch=64)",
    "Model 12 (Adam, 2-Layers(14-ReLU, 7-ReLU), He-Init, BatchNorm, Epochs=50, Batch=64)",
    "Model 13 (Adam-LR(0.0001), 3-Layers(21-LeakyReLU, 14-ReLU, 7-ReLU), He-Init(first 2), Epochs=50, Batch=64)",
    "Model 14 (Adam-LR(0.0001), 2-Layers(14-ReLU, 7-ReLU), Epochs=50, Batch=32)",
    "Model 15 (SGD-Momentum, 3-Layers(21-ReLU, 7-ReLU, 7-ReLU), He-Init, BatchNorm, Epochs=50, Batch=64)"
]


In [ ]:
models_valid_comp_df = pd.concat(
    [
        model_0_valid_perf.T,
        model_1_valid_perf.T,
        model_2_valid_perf.T,
        model_3_valid_perf.T,
        model_4_valid_perf.T,
        model_5_valid_perf.T,
        model_6_valid_perf.T,
        model_7_valid_perf.T,
        model_8_valid_perf.T,
        model_9_valid_perf.T,
        model_10_valid_perf.T,
        model_11_valid_perf.T,
        model_12_valid_perf.T,
        model_13_valid_perf.T,
        model_14_valid_perf.T,
        model_15_valid_perf.T
    ],
    axis=1,
)
models_valid_comp_df.columns = [
    "Model 0 (SGD, 1-Layer(7-ReLU))",
    "Model 1 (SGD-Momentum, 1-Layer(7-ReLU))",
    "Model 2 (SGD-Momentum, 2-Layers(14-ReLU, 7-ReLU))",
    "Model 3 (SGD-Momentum, 2-Layers(14-ReLU, 7-ReLU), Epochs=50)",
    "Model 4 (SGD-Momentum, 2-Layers(14-ReLU, 7-ReLU), Epochs=50, Batch=64)",
    "Model 5 (Adam, 2-Layers(14-ReLU, 7-ReLU), Epochs=50, Batch=64)",
    "Model 6 (Adam, 2-Layers(14-ReLU, 7-ReLU), Dropout(0.4, 0.2), Epochs=50, Batch=64)",
    "Model 7 (Adam, 2-Layers(14-ReLU, 7-ReLU), Dropout(0.4, 0.2), Epochs=50, Batch=32)",
    "Model 8 (Adam, 2-Layers(14-ReLU, 7-ReLU), Epochs=50, Batch=32)",
    "Model 9 (Adam, 2-Layers(14-LeakyReLU, 7-LeakyReLU), Epochs=50, Batch=64)",
    "Model 10 (Adam, 2-Layers(14-ReLU, 7-ReLU), BatchNorm, Epochs=50, Batch=64)",
    "Model 11 (SGD-Momentum, 2-Layers(14-ReLU, 7-ReLU), BatchNorm, Epochs=50, Batch=64)",
    "Model 12 (Adam, 2-Layers(14-ReLU, 7-ReLU), He-Init, BatchNorm, Epochs=50, Batch=64)",
    "Model 13 (Adam-LR(0.0001), 3-Layers(21-LeakyReLU, 14-ReLU, 7-ReLU), He-Init(first 2), Epochs=50, Batch=64)",
    "Model 14 (Adam-LR(0.0001), 2-Layers(14-ReLU, 7-ReLU), Epochs=50, Batch=32)",
    "Model 15 (SGD-Momentum, 3-Layers(21-ReLU, 7-ReLU, 7-ReLU), He-Init, BatchNorm, Epochs=50, Batch=64)"
]


In [ ]:
models_train_comp_df


In [ ]:
models_valid_comp_df


In [ ]:
models_train_comp_df.loc["Recall"] - models_valid_comp_df.loc["Recall"]


<p>Now, let's check the performance of the final model on the test set.</p>


In [ ]:
best_model = model_3


In [ ]:
# Test set performance for the best model
best_model_test_perf = model_performance_classification(best_model, X_test, Y_test)
best_model_test_perf


In [ ]:
# Derive classification_report
Y_test_pred_best = best_model.predict(X_test)

# Check the classification report of best model on test data.
cr_test_best_model = classification_report(Y_test, Y_test_pred_best>0.5)
print(cr_test_best_model)


<h1><strong>Actionable Insights and Recommendations</strong></h1>


<p>Write down some insights and business recommendations based on your observations.</p>


<ol>
<li><p>Model 3 demonstrates a high Recall of 0.87 for identifying actual failures. This is a significant as it directly addresses the most costly outcome: reducing expensive generator replacements (False Negatives). By detecting 87% of impending failures, ReneWind can opt for cheaper repairs, leading to substantial savings.</p>
</li>
<li><p>The Precision of 0.89 for failures indicates that when the model triggers an alert, it is correct nearly 9 out of 10 times. This helps keep inspection costs (False Positives) manageable.</p>
</li>
<li><p>Implement the model within ReneWind's existing operational systems to provide real-time alerts to maintenance teams.</p>
</li>
<li><p>Continuous Monitoring and Retraining can be estalibished. It is crucial to set up a system for Monitoring model performance in production (especially Recall and False Negative rates).</p>
</li>
<li><p>Collecting new failure data and retraining the model periodically to adapt to changing circumstances.</p>
</li>
<li><p>With more data with continous monitoring Renewind can improve the model as it can be retrained with more data points and features and the recall score can be improved from 87% to even higher thereby reducing costs.</p>
</li>
</ol>
